# MAI 600 Module 8 -- Final Capstone
## Local RAG-Grounded PMO Risk Assistant -- Final Version

This notebook documents the real, executed pipeline behind the Module 8 final submission:
chunking, embeddings, vector search, generation, and evaluation, all run against a **local**
Ollama installation on the student's own laptop (not a Colab VM) -- same as Module 7.

**What changed from Module 7:** the knowledge base grew from 6 to 10 documents, the test set
grew from 6 to 10 questions (adding 2 compound questions that require two documents at once),
and retrieval was widened from top-2 to top-4 chunks specifically to fix a compound-question
retrieval failure documented in Module 7. Every number in this notebook is real, captured from
this run -- see `../results/` for the raw CSVs and `../results/summary_metrics.json` for the
summary.


## 1. Load and chunk the knowledge base (10 documents)

In [ ]:
import sys, os
sys.path.insert(0, "../src")
from rag_pipeline import load_chunks, embed, retrieve, build_prompt, generate, cos

KB_DIR = "../data/sample_documents"
all_chunks = load_chunks(KB_DIR)
print(f"Loaded {len(all_chunks)} chunks from {len(os.listdir(KB_DIR))} documents")


## 2. Embed all chunks (real Ollama calls, `nomic-embed-text`)

In [ ]:
for c in all_chunks:
    c["embedding"] = embed(c["text"])

print(f"Embedded {len(all_chunks)} chunks")


## 3. Load the 10 real test questions (8 single-document + 2 compound)

In [ ]:
import pandas as pd

test_cases = pd.read_csv("../data/test_cases.csv")
test_cases[["test_id", "question", "expected_source", "question_type"]]


## 4. Retrieve (top-4), generate, and evaluate for each question (real Ollama calls)

Retrieval was widened from Module 7's top-2 to top-4 specifically to fix a compound-question
failure: a RAID+RACI question previously retrieved 2 RACI chunks and 0 RAID chunks at top-2.

In [ ]:
results = []

for _, row in test_cases.iterrows():
    q_emb = embed(row["question"])
    retrieved = retrieve(q_emb, all_chunks, top_k=4)
    prompt = build_prompt(row["question"], retrieved)
    result = generate(prompt)
    answer = result.get("response", "")
    print(f"{row['test_id']} ({row['question_type']}): retrieved {[c['file'] for c in retrieved]}")
    results.append({
        "test_id": row["test_id"],
        "retrieved_sources": [c["file"] for c in retrieved],
        "answer": answer,
    })

print(f"\nDone -- {len(results)}/10 questions processed")


## 5. Real evaluation summary (loaded from the saved results)

In [ ]:
import json

summary = json.load(open("../results/summary_metrics.json", encoding="utf-8"))
pd.Series(summary)


## 6. Compound-question fix -- before and after

Module 7 documented a real failure: a compound question needing both the RAID log and RACI
matrix documents retrieved 2 RACI chunks and 0 RAID chunks at top-2 retrieval. This project's
evaluation set includes two compound questions (Q9: RAID + RACI, the same failure case
re-tested; Q10: a new vendor-risk + budget-variance scenario). Both retrieved chunks from
**both** required documents at top-4 -- see `../results/retrieved_chunks.csv` and
`../results/evaluation_scores.csv` (Q9, Q10 rows) for the real evidence.

In [ ]:
eval_scores = pd.read_csv("../results/evaluation_scores.csv")
eval_scores[eval_scores["question_type"] == "compound"][
    ["test_id", "question", "expected_source", "retrieval_hit_top4", "citation_match", "observation"]
]


## 7. Honest limitation: two real failures this harder test set surfaced

Not every result improved. `Q10` (a compound question) shows a genuine reasoning error, and an
earlier version of `Q7` was affected by an unquoted-comma bug in the test-case CSV that
truncated the question text sent to the model (fixed and re-run for the results reported here).
Both are disclosed in `evaluation_scores.csv`'s `observation` column rather than hidden.

In [ ]:
eval_scores[eval_scores["accuracy"] <= 3][
    ["test_id", "question_type", "accuracy", "completeness", "observation"]
]


## 8. Fine-tuning readiness (carried forward from Module 7, unchanged)

A real LoRA fine-tuning experiment (`distilgpt2` + `peft`) was executed in Module 7 to test
whether fine-tuning would help this project's format-consistency problem more than RAG already
does. That evidence is carried forward unchanged for this final submission -- it did not need to
be re-run, because RAG's role in this project (grounding factual/policy answers) is a different
capability than what the fine-tuning experiment tested (reproducing a literal output format).
Full detail: `../checkpoints/adapter_or_checkpoint_notes.md`.

In [ ]:
ft_log = json.load(open("../checkpoints/finetune_training_log.json", encoding="utf-8"))
print("Model:", ft_log["model"])
print("Method:", ft_log["method"])
print("Loss: first step =", round(ft_log["loss_history"][0]["loss"], 3),
      "-> last step =", round(ft_log["loss_history"][-1]["loss"], 3))
decision = pd.read_csv("../results/rag_vs_finetuning_decision.csv")
decision[["dimension", "decision_rationale"]]


## 9. Improvement from Module 7 -- full comparison table

See `../results/improvement_comparison.csv` for the complete before/after table with evidence
pointers, and the final article's "Improvement from Prototype" section for the narrative
version.

In [ ]:
pd.read_csv("../results/improvement_comparison.csv")
